In [14]:
import pandas as pd
import torch
import numpy as np
from pathlib import Path
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

#  Configuration 
sns.set_style("whitegrid")
pd.set_option('display.max_columns', None)



In [15]:
# File Paths 
DATA_PROCESSED = Path("../data/processed")
DATA_RAW = Path("../data/raw")

# Check if processed directory exists
if not DATA_PROCESSED.exists():
    raise FileNotFoundError(f"Directory {DATA_PROCESSED} not found.")

## 1. PaySim: Behavioral Aggregation (Transaction -> Node)

In [16]:
paysim_path = DATA_PROCESSED / "paysim_graph_structure.pt"

print(f"Loading PaySim structure from: {paysim_path}")
paysim_struct = torch.load(paysim_path, weights_only=False)

df_sampled = paysim_struct['df_sampled']
account_map = paysim_struct['account_map']
edge_index = paysim_struct['edge_index']
num_nodes = paysim_struct['num_nodes']

print(f"Loaded {len(df_sampled):,} transactions.")
print(f"Graph contains {num_nodes:,} unique nodes (accounts).")

Loading PaySim structure from: ..\data\processed\paysim_graph_structure.pt
Loaded 325,933 transactions.
Graph contains 592,288 unique nodes (accounts).


In [17]:
#  1. Aggregation: As Sender (Source) 
# Group by 'nameOrig'
sender_stats = df_sampled.groupby('nameOrig')['amount'].agg(['count', 'mean', 'max', 'sum'])
sender_stats.columns = ['sent_count', 'sent_amount_mean', 'sent_amount_max', 'sent_amount_sum']

In [18]:
#  2. Aggregation: As Receiver (Destination) 
# Group by 'nameDest'
receiver_stats = df_sampled.groupby('nameDest')['amount'].agg(['count', 'mean', 'max', 'sum'])
receiver_stats.columns = ['rcvd_count', 'rcvd_amount_mean', 'rcvd_amount_max', 'rcvd_amount_sum']

In [19]:
#  3. Create the Node DataFrame 
node_df = pd.DataFrame(list(account_map.items()), columns=['accountID', 'node_idx'])
node_df.set_index('accountID', inplace=True)

# Merge stats onto the node list
node_features_df = node_df.join(sender_stats).join(receiver_stats)

# Fill NaNs with 0 (e.g., if a node never sent money, sent_amount is 0)
node_features_df.fillna(0, inplace=True)

# Sort by node_idx to ensure the matrix rows align perfectly with the graph tensor logic
node_features_df = node_features_df.sort_values('node_idx')

print(f"Node feature matrix shape: {node_features_df.shape}")
print("\nFirst 5 rows of aggregated features:")
node_features_df.head()

Node feature matrix shape: (592288, 9)

First 5 rows of aggregated features:


,node_idx,sent_count,sent_amount_mean,sent_amount_max,sent_amount_sum,rcvd_count,rcvd_amount_mean,rcvd_amount_max,rcvd_amount_sum
accountID,,,,,,,,,
C1611144363,0,1.0,10000000.00,10000000.00,10000000.00,0.0,0.0,0.0,0.0
C1409142151,1,1.0,320909.86,320909.86,320909.86,0.0,0.0,0.0,0.0
C194892485,2,1.0,148813.54,148813.54,148813.54,0.0,0.0,0.0,0.0
C1873000881,3,1.0,222611.57,222611.57,222611.57,0.0,0.0,0.0,0.0
C55146780,4,1.0,5563.80,5563.80,5563.80,0.0,0.0,0.0,0.0


In [20]:
# Identify accounts that initiated fraud
fraud_senders = df_sampled[df_sampled['isFraud'] == 1]['nameOrig'].unique()
fraud_set = set(fraud_senders)

print(f"Number of unique accounts that initiated fraud in this sample: {len(fraud_set):,}")

# Create the label vector
node_labels = [1 if acct in fraud_set else 0 for acct in node_features_df.index]

# Convert to Tensor
y_paysim = torch.tensor(node_labels, dtype=torch.long)

print(f"Label tensor created. Shape: {y_paysim.shape}")
print(f"Fraudulent Nodes (Class 1): {y_paysim.sum().item()}")
print(f"Legitimate Nodes (Class 0): {(y_paysim == 0).sum().item()}")

Number of unique accounts that initiated fraud in this sample: 8,213
Label tensor created. Shape: torch.Size([592288])
Fraudulent Nodes (Class 1): 8213
Legitimate Nodes (Class 0): 584075


In [21]:
# Extract feature columns 
feature_cols = ['sent_count', 'sent_amount_mean', 'sent_amount_max', 'sent_amount_sum',
                'rcvd_count', 'rcvd_amount_mean', 'rcvd_amount_max', 'rcvd_amount_sum']

x_numpy = node_features_df[feature_cols].values
x_paysim = torch.tensor(x_numpy, dtype=torch.float)

# Create the PyG Data object
paysim_data = Data(x=x_paysim, edge_index=edge_index, y=y_paysim)
paysim_data.num_nodes = num_nodes

print("PaySim Data Object Constructed:")
print(paysim_data)

PaySim Data Object Constructed:
Data(x=[592288, 8], edge_index=[2, 325933], y=[592288], num_nodes=592288)


## 2. Global Feature Standardization

In [24]:
def standardize_graph_data(data_obj, name="Graph"):
    """
    Applies StandardScaler to data_obj.x and updates the object.
    Returns the scaler for reference if needed.
    """
    print(f"\n Processing {name} ")
    print(f"Original Mean (first 5 feats): {data_obj.x.mean(dim=0)[:5].numpy().round(2)}")
    print(f"Original Std  (first 5 feats): {data_obj.x.std(dim=0)[:5].numpy().round(2)}")
    
    scaler = StandardScaler()
    
    # Convert to numpy, fit_transform, convert back to torch float
    x_scaled = scaler.fit_transform(data_obj.x.numpy())
    data_obj.x = torch.tensor(x_scaled, dtype=torch.float)
    
    print(f"New Mean (approx 0): {data_obj.x.mean(dim=0)[:5].numpy().round(2)}")
    print(f"New Std  (approx 1): {data_obj.x.std(dim=0)[:5].numpy().round(2)}")
    
    return data_obj

In [25]:
#  1. Standardize PaySim 
paysim_data = standardize_graph_data(paysim_data, name="PaySim")

#  2. Standardize Elliptic 
elliptic_path = DATA_PROCESSED / "elliptic_graph.pt"
if elliptic_path.exists():
    elliptic_data = torch.load(elliptic_path, weights_only=False)
    elliptic_data = standardize_graph_data(elliptic_data, name="Elliptic")
else:
    print("Warning: Elliptic graph not found. Skipping.")

#  3. Standardize Synthetic 
synthetic_path = DATA_PROCESSED / "synthetic_fraud_clean.pt"
if synthetic_path.exists():
    synthetic_data = torch.load(synthetic_path, weights_only=False)
    synthetic_data = standardize_graph_data(synthetic_data, name="Synthetic")
else:
    print("Warning: Synthetic graph not found. Skipping.")


 Processing PaySim 
Original Mean (first 5 feats): [ 0.  0. -0.  0. -0.]
Original Std  (first 5 feats): [1. 1. 1. 1. 1.]
New Mean (approx 0): [ 0.  0. -0.  0.  0.]
New Std  (approx 1): [1. 1. 1. 1. 1.]

 Processing Elliptic 
Original Mean (first 5 feats): [ 0. -0.  0.  0. -0.]
Original Std  (first 5 feats): [1. 1. 1. 1. 1.]
New Mean (approx 0): [ 0. -0.  0.  0. -0.]
New Std  (approx 1): [1. 1. 1. 1. 1.]

 Processing Synthetic 
Original Mean (first 5 feats): [  8.18   8.18 779.71 779.71  81.88]
Original Std  (first 5 feats): [  15.29    2.83 1605.19  377.31   52.22]
New Mean (approx 0): [-0. -0. -0. -0. -0.]
New Std  (approx 1): [1. 1. 1. 1. 1.]


## 3. Final Validation & Export

In [26]:
print("\n Final Dataset Summary ")

datasets = {
    "PaySim": paysim_data,
    "Elliptic": elliptic_data if 'elliptic_data' in locals() else None,
    "Synthetic": synthetic_data if 'synthetic_data' in locals() else None
}

for name, data in datasets.items():
    if data is not None:
        print(f"\ndataset: {name}")
        print(f"  Nodes: {data.num_nodes:,}")
        print(f"  Edges: {data.edge_index.shape[1]:,}")
        print(f"  Features (x): {data.x.shape}")
        print(f"  Labels (y):   {data.y.shape}")
        
        # Save the finalized version
        save_name = f"{name.lower()}_graph_v2.pt"
        torch.save(data, DATA_PROCESSED / save_name)
        print(f"   Saved to {DATA_PROCESSED / save_name}")

print("\nAll datasets feature-engineered and standardized successfully.")


 Final Dataset Summary 

dataset: PaySim
  Nodes: 592,288
  Edges: 325,933
  Features (x): torch.Size([592288, 8])
  Labels (y):   torch.Size([592288])
   Saved to ..\data\processed\paysim_graph_v2.pt

dataset: Elliptic
  Nodes: 203,769
  Edges: 234,355
  Features (x): torch.Size([203769, 165])
  Labels (y):   torch.Size([203769])
   Saved to ..\data\processed\elliptic_graph_v2.pt

dataset: Synthetic
  Nodes: 5,000
  Edges: 40,900
  Features (x): torch.Size([5000, 12])
  Labels (y):   torch.Size([5000])
   Saved to ..\data\processed\synthetic_graph_v2.pt

All datasets feature-engineered and standardized successfully.
